# minnode — Data Preparation and Algorithm Validation

This notebook prepares the data consumed by the `minnode` dashboard and validates the
shortest-path algorithm used to compute optimal meeting destinations.

## Sections

1. **Data wrangling** — Filter raw GSA City Pair fare data to U.S. CONUS airports,
   join with airport coordinates, and write a single denormalized CSV that the
   dashboard consumes.
2. **Algorithm validation** — Build a graph from the produced CSV, implement
   Dijkstra's shortest-path algorithm, and verify it computes optimal meeting
   points for representative inputs.

## Inputs

- `../data/awards_2026.csv` — GSA City Pair FY 2026 awards (16,073 records).
- `../data/iata-icao.csv` — Airport reference data from `ip2location/ip2location-iata-icao` (CC BY-SA 4.0).

## Output

- `../data/processed/airport_pair_fares.csv` — Denormalized airport-pair fares
  with origin and destination coordinates, filtered to U.S. CONUS.


## Section 1: Data Wrangling

In [1]:
import pandas as pd

awards = pd.read_csv('../data/awards_2026.csv')
airports = pd.read_csv('../data/iata-icao.csv')

print(f"City Pair awards: {len(awards):,} records")
print(f"Airport reference: {len(airports):,} records")

City Pair awards: 16,073 records
Airport reference: 9,158 records


### Filter awards to U.S. CONUS routes

GSA City Pair includes international markets, indicated by metropolitan codes
(e.g., `LON`, `NYC`) that do not correspond to single airports. The dashboard's
scope is U.S. CONUS travel, so international markets and non-CONUS U.S. routes
(Alaska, Hawaii, territories) are excluded.

In [2]:
awards_us = awards[
    (awards['ORIGIN_COUNTRY'] == 'UNITED STATES') &
    (awards['DESTINATION_COUNTRY'] == 'UNITED STATES')
].copy()

print(f"U.S. domestic awards: {len(awards_us):,} records")

U.S. domestic awards: 11,692 records


### Filter airport reference to CONUS

The airport reference dataset is global. Filter to U.S. airports excluding
Alaska and Hawaii (the territories not present in the dataset by default).

In [3]:
airports_conus = airports[
    (airports['country_code'] == 'US') &
    (~airports['region_name'].isin(['Alaska', 'Hawaii']))
].copy()

# Keep only valid 3-letter IATA codes with parsable coordinates.
airports_conus = airports_conus[
    airports_conus['iata'].notna() &
    (airports_conus['iata'].str.len() == 3)
]
airports_conus['latitude'] = pd.to_numeric(airports_conus['latitude'], errors='coerce')
airports_conus['longitude'] = pd.to_numeric(airports_conus['longitude'], errors='coerce')
airports_conus = airports_conus.dropna(subset=['latitude', 'longitude'])

# Keep one row per IATA code with the columns we need.
airports_conus = (
    airports_conus[['iata', 'latitude', 'longitude']]
    .drop_duplicates(subset='iata')
    .rename(columns={'iata': 'code', 'latitude': 'lat', 'longitude': 'lon'})
)

print(f"CONUS airports with coordinates: {len(airports_conus):,}")

CONUS airports with coordinates: 1,675


### Join awards with airport coordinates

Produce a single denormalized table: one row per directed airport-pair, with
fare columns and coordinates for both origin and destination.

In [4]:
fare_columns = ['YCA_FARE', '_CA_FARE', 'BUSINESS_FARE', '_CP_FARE']

pairs = awards_us[[
    'ORIGIN_AIRPORT_ABBREV',
    'DESTINATION_AIRPORT_ABBREV',
    *fare_columns,
]].rename(columns={
    'ORIGIN_AIRPORT_ABBREV': 'origin',
    'DESTINATION_AIRPORT_ABBREV': 'destination',
    'YCA_FARE': 'yca_fare',
    '_CA_FARE': 'ca_fare',
    'BUSINESS_FARE': 'business_fare',
    '_CP_FARE': 'cp_fare',
})

# Join origin coordinates.
pairs = pairs.merge(
    airports_conus.rename(columns={'code': 'origin', 'lat': 'origin_lat', 'lon': 'origin_lon'}),
    on='origin',
    how='inner',
)

# Join destination coordinates.
pairs = pairs.merge(
    airports_conus.rename(columns={'code': 'destination', 'lat': 'destination_lat', 'lon': 'destination_lon'}),
    on='destination',
    how='inner',
)

# Order columns for the output schema.
pairs = pairs[[
    'origin', 'origin_lat', 'origin_lon',
    'destination', 'destination_lat', 'destination_lon',
    'yca_fare', 'ca_fare', 'business_fare', 'cp_fare',
]]

print(f"Airport-pair fare records: {len(pairs):,}")
pairs.head()

Airport-pair fare records: 10,935


,origin,origin_lat,origin_lon,destination,destination_lat,destination_lon,yca_fare,ca_fare,business_fare,cp_fare
0,ABE,40.6521,-75.4408,ATL,33.6367,-84.4281,676,338,0,0
1,ABE,40.6521,-75.4408,CLT,35.2140,-80.9431,325,0,0,0
2,ABE,40.6521,-75.4408,DCA,38.8522,-77.0378,280,0,0,0
3,ABE,40.6521,-75.4408,ORD,41.9786,-87.9047,619,310,0,0
4,ABI,32.4113,-99.6819,DCA,38.8522,-77.0378,455,0,0,0


### Write the dashboard CSV

In [5]:
import os
os.makedirs('../data/processed', exist_ok=True)

output_path = '../data/processed/airport_pair_fares.csv'
pairs.to_csv(output_path, index=False)
print(f"Wrote {output_path}")

Wrote ../data/processed/airport_pair_fares.csv


## Section 2: Algorithm Validation

Build a graph from the produced CSV, implement Dijkstra's shortest-path
algorithm, and verify it computes the optimal meeting destination for a set of
origins.

The algorithm used here is Dijkstra's, not A\*. A\* requires an admissible
heuristic; identifying a suitable heuristic for fare-graph search is deferred to
a later iteration. Implemented as written, the algorithm is mathematically
equivalent to Dijkstra.

### Build the graph

Edges are undirected. The fare column used here is `yca_fare` (the unrestricted
government coach fare); the choice of fare column is a modeling decision
documented elsewhere in the framework.

In [6]:
graph = {}
for _, row in pairs.iterrows():
    origin = row['origin']
    destination = row['destination']
    fare = row['yca_fare']

    graph.setdefault(origin, []).append((destination, fare))
    graph.setdefault(destination, []).append((origin, fare))

print(f"Graph nodes: {len(graph):,}")
print(f"Graph edges (directed): {sum(len(v) for v in graph.values()):,}")

Graph nodes: 289
Graph edges (directed): 21,870


### Dijkstra's algorithm

Compute the shortest path from a starting airport to every other reachable airport
in the graph. Returns the cost map (airport → minimum cost from start).

In [7]:
import heapq
import math

def dijkstra(graph, start):
    """Compute shortest-path costs from `start` to every reachable node."""
    costs = {node: math.inf for node in graph}
    costs[start] = 0
    queue = [(0, start)]

    while queue:
        current_cost, current_node = heapq.heappop(queue)
        if current_cost > costs[current_node]:
            continue
        for neighbor, weight in graph[current_node]:
            new_cost = current_cost + weight
            if new_cost < costs[neighbor]:
                costs[neighbor] = new_cost
                heapq.heappush(queue, (new_cost, neighbor))

    return costs

### Compute optimal meeting point

For a set of origin airports, find the destination that minimizes the sum of
shortest-path costs from each origin. This is the artifact's core computation.

In [8]:
def optimal_meeting_point(graph, origins):
    """Return (best_destination, total_cost, per_origin_costs)."""
    cost_maps = {origin: dijkstra(graph, origin) for origin in origins}

    candidates = set(graph.keys())
    best_destination = None
    best_total = math.inf

    for candidate in candidates:
        total = sum(cost_maps[origin].get(candidate, math.inf) for origin in origins)
        if total < best_total:
            best_total = total
            best_destination = candidate

    per_origin = {origin: cost_maps[origin][best_destination] for origin in origins}
    return best_destination, best_total, per_origin

### Validation: representative example

In [9]:
origins = ['MSP', 'DCA', 'LAX']
destination, total_cost, per_origin = optimal_meeting_point(graph, origins)

print(f"Origins: {origins}")
print(f"Optimal meeting destination: {destination}")
print(f"Total cost: ${total_cost:,.2f}")
print()
print("Per-origin costs:")
for origin, cost in per_origin.items():
    print(f"  {origin} -> {destination}: ${cost:,.2f}")

Origins: ['MSP', 'DCA', 'LAX']
Optimal meeting destination: ORD
Total cost: $352.00

Per-origin costs:
  MSP -> ORD: $89.00
  DCA -> ORD: $114.00
  LAX -> ORD: $149.00


### Ranked candidates

Compute the total cost for every candidate destination and display the top
candidates ranked by total cost. Held in memory; not written to disk.

In [10]:
cost_maps = {origin: dijkstra(graph, origin) for origin in origins}

candidates = []
for candidate in graph.keys():
    total = sum(cost_maps[origin].get(candidate, math.inf) for origin in origins)
    if math.isfinite(total):
        candidates.append((candidate, total))

ranked = pd.DataFrame(
    sorted(candidates, key=lambda x: x[1]),
    columns=['destination', 'total_cost'],
)
ranked.head(10)

,destination,total_cost
0,ORD,352
1,MSP,425
2,DEN,448
3,DCA,466
4,EWR,481
5,LAX,485
6,SAT,517
7,BOS,522
8,GRR,529
9,MCI,539
